# News Encoder → Survey Expectation Pipeline

**Goal:** Calibrate a frozen FinBERT encoder + Ridge regression to map aggregated
financial-news embeddings to median survey forecasts (e.g.\ SPF 1-year CPI),
then extrapolate the fitted map backward through the historical news record.

**Pipeline stages:**

1. Load news corpus and survey data
2. Encode articles with frozen FinBERT (`ProsusAI/finbert`)
3. Aggregate article embeddings within each survey wave window
4. Fit Ridge regression (LOO-CV over waves) on the calibration period
5. Evaluate on a temporal hold-out
6. Apply fitted map to pre-calibration / historical news
7. Diagnostics: support check via Mahalanobis distance in PCA space

**TODO markers** indicate cells where you need to plug in your data or tune
hyperparameters. Everything else should run as-is once data is loaded.

## 1. Setup

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import LeaveOneOut
from sklearn.decomposition import PCA
from sklearn.covariance import LedoitWolf

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/aaron/opt/anaconda3/envs/news-encoder/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/aaron/opt/anaconda3/envs/news-encoder/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/aaron/opt/anaconda3/envs/news-encoder/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/aaron/opt/anaconda3/envs/news-encoder/lib/python3.10/site-packages/t

Using device: cpu


## 2. Configuration

**TODO:** review and tune as needed.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────
DATA_DIR     = Path("./data")           # TODO: point to your data directory
CACHE_DIR    = Path("./cache")          # embeddings will be cached here
OUTPUT_DIR   = Path("./output")         # final predictions, plots, model
for d in [CACHE_DIR, OUTPUT_DIR]:
    d.mkdir(exist_ok=True, parents=True)

# ── Encoder ──────────────────────────────────────────────────────────
ENCODER_NAME   = "ProsusAI/finbert"     # frozen throughout
MAX_TOKEN_LEN  = 512                    # BERT context window
ENCODE_BATCH   = 32                     # adjust for GPU memory

# ── Aggregation ──────────────────────────────────────────────────────
AGG_WINDOW_DAYS  = 90                   # days of news before survey closing date
RECENCY_WEIGHTING = "uniform"           # "uniform" | "exponential"
RECENCY_HALFLIFE_DAYS = 7               # used if exponential

# ── Calibration / hold-out split ────────────────────────────────────
HOLDOUT_FRAC   = 0.20                   # fraction of waves held out (most recent)

# ── Ridge ────────────────────────────────────────────────────────────
RIDGE_ALPHAS   = np.logspace(-3, 4, 50)

# ── Diagnostics ──────────────────────────────────────────────────────
PCA_COMPONENTS_FOR_MAHALANOBIS = 30     # for support check in low-dim space

## 3. Load News Corpus

**TODO:** plug in your news data here. Expected schema:

| column   | type        | description                              |
|----------|-------------|------------------------------------------|
| `date`   | datetime    | publication date                         |
| `text`   | str         | article body (or headline + lede)        |
| `source` | str (opt.)  | publisher, e.g.\ "WSJ", "FT", "Bloomberg" |

The script below is a placeholder that creates a small dummy DataFrame so you can
verify the pipeline end-to-end before plugging in real data. **Replace this cell**
with your actual data load (CSV, Parquet, SQL, etc.).

In [1]:
# TODO: set the path to your parquet file
NEWS_PARQUET = DATA_DIR / "your_news_file.parquet"   # <-- fill in filename

# TODO: set the column names as they appear in your parquet
NEWS_DATE_COL   = "date"    # column containing article publication date
NEWS_TEXT_COL   = "text"    # column containing article body / lede
NEWS_SOURCE_COL = None      # column for publisher, or None if absent


def load_news_corpus():
    '''Load news corpus from parquet. Returns DataFrame[date, text].'''
    df = pd.read_parquet(NEWS_PARQUET)

    # Rename to standard column names used downstream
    rename = {NEWS_DATE_COL: "date", NEWS_TEXT_COL: "text"}
    if NEWS_SOURCE_COL:
        rename[NEWS_SOURCE_COL] = "source"
    df = df.rename(columns=rename)[list(rename.values())]

    # Ensure datetime dtype (parquet usually preserves this, but just in case)
    df["date"] = pd.to_datetime(df["date"])

    # Drop rows with missing text or date
    df = df.dropna(subset=["date", "text"]).reset_index(drop=True)

    return df.sort_values("date").reset_index(drop=True)


news_df = load_news_corpus()
print(f"Loaded {len(news_df):,} articles")
print(f"Date range: {news_df['date'].min().date()} → {news_df['date'].max().date()}")
print(f"Columns: {list(news_df.columns)}")
news_df.head()

NameError: name 'DATA_DIR' is not defined

## 4. Load Survey Data

Two series, both quarterly:
- **Dividend growth expectations** (`Dividend_growth_expectations.xlsx`) — 2003 onward
- **Earnings growth expectations** (`Earnings_growth_expectations.xlsx`) — 1976 onward

Both have `Year` and `Quarter` columns rather than a single date column, so we
construct `wave_date` by combining them. The pipeline will loop over both series.

In [ ]:
# TODO: update filenames if yours differ
DIVIDEND_EXCEL = DATA_DIR / "Dividend_growth_expectations.xlsx"
EARNINGS_EXCEL = DATA_DIR / "Earnings_growth_expectations.xlsx"


def make_wave_date(df):
    '''Combine Year + Quarter columns into a datetime (start of quarter).'''
    return pd.to_datetime(
        df["Year"].astype(int).astype(str) + "Q" + df["Quarter"].astype(int).astype(str)
    )


def load_dividend_expectations():
    df = pd.read_excel(DIVIDEND_EXCEL)
    return pd.DataFrame({
        "wave_date":       make_wave_date(df),
        "median_forecast": pd.to_numeric(df["Expected one-year log dividend growth"], errors="coerce"),
    }).dropna().sort_values("wave_date").reset_index(drop=True)


def load_earnings_expectations():
    # Earnings file has a two-row merged header; skip the first header row
    # and use the second row as column names, then take the first denominator
    # variant (current earnings e_t) — columns 2 and 3 (0-indexed after Year/Quarter)
    df = pd.read_excel(EARNINGS_EXCEL, header=[0, 1])
    # Flatten multi-index columns
    df.columns = [
        " | ".join(str(c).strip() for c in col if "Unnamed" not in str(c)).strip(" | ")
        for col in df.columns
    ]
    # Rename Year/Quarter which land as the first two columns
    df = df.rename(columns={df.columns[0]: "Year", df.columns[1]: "Quarter"})
    # The first earnings variant is the third column
    forecast_col = df.columns[2]
    return pd.DataFrame({
        "wave_date":       make_wave_date(df),
        "median_forecast": pd.to_numeric(df[forecast_col], errors="coerce"),
    }).dropna().sort_values("wave_date").reset_index(drop=True)


# Dict of all survey series — add more here if needed
SURVEY_SERIES = {
    "dividend_growth":  load_dividend_expectations,
    "earnings_growth":  load_earnings_expectations,
}

# Quick preview
for name, loader in SURVEY_SERIES.items():
    df = loader()
    print(f"{name}: {len(df)} waves, "
          f"{df['wave_date'].min().date()} → {df['wave_date'].max().date()}, "
          f"forecast range [{df['median_forecast'].min():.3f}, {df['median_forecast'].max():.3f}]")

## 5. Load FinBERT Encoder

We use the encoder *only* — no fine-tuning, no sentiment head. We extract the
`[CLS]` embedding from the last hidden state as the document representation.

In [ ]:
print(f"Loading {ENCODER_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
model = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE)
model.eval()
for p in model.parameters():
    p.requires_grad = False
print(f"  hidden size = {model.config.hidden_size}")
print(f"  vocab size  = {model.config.vocab_size}")

## 6. Encode Articles

We batch articles through FinBERT and extract the `[CLS]` embedding (768-dim).
Articles longer than `MAX_TOKEN_LEN=512` tokens are truncated — for news, the
lede typically carries the macro signal, so this is acceptable. If you want to
encode full long-form articles, replace `encode_articles_batch` with a chunking
version that mean-pools embeddings across 512-token chunks.

Embeddings are cached to disk so you don't re-encode on every run.

In [ ]:
def encode_articles_batch(texts, batch_size=ENCODE_BATCH, max_length=MAX_TOKEN_LEN):
    '''Encode a list of strings. Returns (n, hidden_size) numpy array.'''
    embeddings = []
    n = len(texts)
    for i in tqdm(range(0, n, batch_size), desc="Encoding"):
        batch = list(texts[i:i+batch_size])
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs)
        # [CLS] token from last hidden state
        cls_emb = out.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32)
        embeddings.append(cls_emb)
    return np.vstack(embeddings)

In [ ]:
# Cache embeddings keyed by corpus size + first/last date as sanity hash
cache_key = f"news_emb_n{len(news_df)}_{news_df['date'].min():%Y%m%d}_{news_df['date'].max():%Y%m%d}.npy"
cache_path = CACHE_DIR / cache_key

if cache_path.exists():
    print(f"Loading cached embeddings from {cache_path}")
    article_embeddings = np.load(cache_path)
else:
    print(f"Encoding {len(news_df):,} articles...")
    article_embeddings = encode_articles_batch(news_df["text"].tolist())
    np.save(cache_path, article_embeddings)
    print(f"Cached to {cache_path}")

print(f"article_embeddings shape: {article_embeddings.shape}")

## 7–13. Pipeline Loop Over Survey Series

Everything from aggregation through saving runs once per survey series.
Results are stored in `all_results` (a dict keyed by series name) and
written to separate subdirectories under `OUTPUT_DIR`.

In [ ]:
def recency_weights(article_dates, wave_date, scheme="uniform", halflife_days=7):
    '''Return per-article weights for aggregation within a wave window.'''
    days_back = (wave_date - article_dates).dt.days.values
    if scheme == "uniform":
        return np.ones(len(article_dates))
    elif scheme == "exponential":
        return 0.5 ** (days_back / halflife_days)
    else:
        raise ValueError(f"Unknown scheme: {scheme}")


def aggregate_to_waves(news_df, article_embeddings, survey_df,
                       window_days=AGG_WINDOW_DAYS,
                       scheme=RECENCY_WEIGHTING,
                       halflife=RECENCY_HALFLIFE_DAYS):
    '''Returns (Z, valid_waves_df) where Z is (n_waves, hidden_size).'''
    Z, valid_idx, n_articles_per_wave = [], [], []
    for i, row in survey_df.iterrows():
        wd = row["wave_date"]
        mask = (
            (news_df["date"] >= wd - pd.Timedelta(days=window_days)) &
            (news_df["date"] < wd)
        )
        idx = np.where(mask)[0]
        if len(idx) == 0:
            continue
        emb = article_embeddings[idx]
        w = recency_weights(news_df.loc[idx, "date"], wd, scheme, halflife)
        z = (emb * w[:, None]).sum(axis=0) / w.sum()
        Z.append(z)
        valid_idx.append(i)
        n_articles_per_wave.append(len(idx))

    Z = np.vstack(Z)
    valid_df = survey_df.iloc[valid_idx].copy().reset_index(drop=True)
    valid_df["n_articles"] = n_articles_per_wave
    return Z, valid_df


def run_pipeline(series_name, survey_loader, news_df, article_embeddings):
    '''
    Full pipeline for one survey series. Returns a dict of results.
    Saves outputs to OUTPUT_DIR / series_name /.
    '''
    print(f"\n{'='*60}")
    print(f"  Running pipeline: {series_name}")
    print(f"{'='*60}")

    out_dir = OUTPUT_DIR / series_name
    out_dir.mkdir(exist_ok=True, parents=True)

    # ── Load survey ──────────────────────────────────────────────────
    survey_df = survey_loader()
    print(f"Survey waves: {len(survey_df)} "
          f"({survey_df['wave_date'].min().date()} → {survey_df['wave_date'].max().date()})")

    # ── Aggregate to waves ───────────────────────────────────────────
    Z, waves_df = aggregate_to_waves(news_df, article_embeddings, survey_df)
    y = waves_df["median_forecast"].values
    print(f"Waves with news coverage: {len(waves_df)} "
          f"(articles/wave: median={waves_df['n_articles'].median():.0f}, "
          f"min={waves_df['n_articles'].min()})")

    # ── Train / hold-out split ───────────────────────────────────────
    n_waves   = len(waves_df)
    n_holdout = max(1, int(round(HOLDOUT_FRAC * n_waves)))
    n_calib   = n_waves - n_holdout

    Z_cal,  Z_hold  = Z[:n_calib],       Z[n_calib:]
    y_cal,  y_hold  = y[:n_calib],       y[n_calib:]
    w_cal,  w_hold  = waves_df.iloc[:n_calib], waves_df.iloc[n_calib:]
    print(f"Calibration: {n_calib} waves | Hold-out: {n_holdout} waves")

    # ── Ridge with LOO-CV ────────────────────────────────────────────
    ridge = RidgeCV(alphas=RIDGE_ALPHAS, cv=LeaveOneOut(),
                    scoring="neg_mean_squared_error")
    ridge.fit(Z_cal, y_cal)
    print(f"Ridge α: {ridge.alpha_:.4g} | In-sample R²: {ridge.score(Z_cal, y_cal):.3f}")

    # ── Hold-out evaluation ──────────────────────────────────────────
    y_pred_cal  = ridge.predict(Z_cal)
    y_pred_hold = ridge.predict(Z_hold)

    rmse_hold = np.sqrt(np.mean((y_hold - y_pred_hold) ** 2))
    dy_true   = np.diff(y_hold)
    dy_pred   = np.diff(y_pred_hold)
    dir_acc   = np.mean(np.sign(dy_true) == np.sign(dy_pred)) if len(dy_true) > 0 else np.nan
    y_ar1     = np.concatenate([[y_cal[-1]], y_hold[:-1]])
    rmse_ar1  = np.sqrt(np.mean((y_hold - y_ar1) ** 2))

    print(f"Hold-out RMSE — FinBERT Ridge: {rmse_hold:.4f} | AR(1): {rmse_ar1:.4f}")
    print(f"Directional accuracy: {dir_acc:.1%}")

    # ── Mahalanobis support check ────────────────────────────────────
    n_pca = min(PCA_COMPONENTS_FOR_MAHALANOBIS, n_calib - 1)
    pca   = PCA(n_components=n_pca, random_state=SEED)
    Z_cal_pca  = pca.fit_transform(Z_cal)
    Z_hold_pca = pca.transform(Z_hold)

    lw      = LedoitWolf().fit(Z_cal_pca)
    mu      = Z_cal_pca.mean(axis=0)
    inv_cov = np.linalg.pinv(lw.covariance_)

    def mahal(z_pca):
        diff = z_pca - mu
        return np.sqrt(np.einsum("ij,jk,ik->i", diff, inv_cov, diff))

    d_cal  = mahal(Z_cal_pca)
    d_hold = mahal(Z_hold_pca)
    threshold = np.quantile(d_cal, 0.99)

    # ── Plots ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    fig.suptitle(series_name.replace("_", " ").title(), fontweight="bold")

    ax = axes[0]
    ax.scatter(y_cal,  y_pred_cal,  alpha=0.5, label="Calibration", color="C0")
    ax.scatter(y_hold, y_pred_hold, alpha=0.8, label="Hold-out",    color="C3")
    lo, hi = min(y.min(), y_pred_cal.min()), max(y.max(), y_pred_cal.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="45°")
    ax.set_xlabel("Realized"); ax.set_ylabel("Predicted")
    ax.set_title("Predicted vs. realized"); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.plot(w_cal["wave_date"],  y_cal,       "-",  color="C0", label="Realized (cal)")
    ax.plot(w_cal["wave_date"],  y_pred_cal,  "--", color="C0", alpha=0.7)
    ax.plot(w_hold["wave_date"], y_hold,      "-",  color="C3", label="Realized (hold)")
    ax.plot(w_hold["wave_date"], y_pred_hold, "--", color="C3", alpha=0.7, label="Predicted (hold)")
    ax.axvline(w_hold["wave_date"].iloc[0], color="gray", ls=":", alpha=0.7)
    ax.set_xlabel("Wave date"); ax.set_ylabel("Forecast")
    ax.set_title("Time series fit"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(out_dir / "fit_diagnostics.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── Save ─────────────────────────────────────────────────────────
    results_df = pd.concat([
        pd.DataFrame({"wave_date": w_cal["wave_date"],  "realized": y_cal,
                      "predicted": y_pred_cal,  "mahal_dist": d_cal,
                      "out_of_support": d_cal > threshold, "split": "calibration"}),
        pd.DataFrame({"wave_date": w_hold["wave_date"], "realized": y_hold,
                      "predicted": y_pred_hold, "mahal_dist": d_hold,
                      "out_of_support": d_hold > threshold, "split": "holdout"}),
    ], ignore_index=True)
    results_df.to_csv(out_dir / "predictions.csv", index=False)

    with open(out_dir / "model.pkl", "wb") as f:
        pickle.dump({
            "ridge": ridge, "pca": pca, "ledoit_wolf": lw,
            "calibration_mean": mu, "support_threshold": threshold,
            "config": {
                "series": series_name, "encoder": ENCODER_NAME,
                "agg_window_days": AGG_WINDOW_DAYS, "ridge_alpha": float(ridge.alpha_),
                "n_calib": int(n_calib), "n_holdout": int(n_holdout),
            },
        }, f)

    print(f"Saved to {out_dir}/")
    return {
        "series": series_name, "ridge": ridge, "pca": pca,
        "waves_df": waves_df, "results_df": results_df,
        "rmse_hold": rmse_hold, "rmse_ar1": rmse_ar1, "dir_acc": dir_acc,
    }

In [ ]:
# Run the full pipeline for every survey series
all_results = {}
for series_name, loader in SURVEY_SERIES.items():
    all_results[series_name] = run_pipeline(
        series_name, loader, news_df, article_embeddings
    )

In [ ]:
# Summary table across series
print("\n── Summary ──────────────────────────────────────────")
print(f"{'Series':<25} {'RMSE (Ridge)':>14} {'RMSE (AR1)':>12} {'Dir Acc':>10}")
print("-" * 65)
for name, res in all_results.items():
    print(f"{name:<25} {res['rmse_hold']:>14.4f} {res['rmse_ar1']:>12.4f} {res['dir_acc']:>9.1%}")

## Historical Extrapolation

**TODO:** Once you have the full news corpus, load it here and call
`run_pipeline` again or extend `run_pipeline` to accept a historical
news DataFrame. The `model.pkl` saved above contains everything needed
(`ridge`, `pca`, fitted means) to call `ridge.predict()` on any new
aggregated embedding matrix.

For now with the PoC sample (1965–2014), the news corpus already covers
the full period so the calibration results above are your extrapolation.